In [35]:
import logging

import pandas as pd

In [142]:
PRODUCER_COL_NAME: str = "producer" # col for wich producer 1 to 4
DURATION_PRODUCER_COL_NAME: str = "duration_producer" # col for duration from producer
DURATION_COATING_COL_NAME: str = "duration_coating" # col for duration of coating
DURATION_COL_NAME: str = "duration" # whole production duration
PRODUCTION_LINE_COL_NAME: str = "production_line" # The production line

In [36]:
logger = logging.getLogger(__name__)

In [20]:
def show_only_producer_cols(input):
    return input[["SKU", PRODUCER_COL_NAME]]


In [41]:
def clean_producer(input):
    # Get the producer cols
    producer_cols = [c for c in input.columns if c.startswith("Produzent")]

    # map producer cols to producer
    input[PRODUCER_COL_NAME] = input[producer_cols].idxmax(axis=1).str.extract(r"(\d+)").astype(float)

    # fill NaNs with 0
    input[PRODUCER_COL_NAME] = input[PRODUCER_COL_NAME].fillna(0).astype(int)

    # Check that no field is NaN
    assert(len(input[input[PRODUCER_COL_NAME] == 0]) == 0)

    # Drop the producer cols
    try:
        input = input.drop(columns=producer_cols)
    except KeyError:
        logger.debug("cols already dropped")

    return input

In [42]:
shipping_time_1 = pd.read_excel("/Users/ap8c3/IdeaProjects/Hackathon/eleo-mind/packages/backend/files/Produktionszeit_Warenfluss_Hackathon_Teilnehmer.xlsx", sheet_name=1)

cleaned_up_shipping_time = clean_producer(shipping_time_1)

cleaned_up_shipping_time
# show_only_producer_cols(cleaned_up_shipping_time)


,SKU,Beschichtung,Duplexbeschichter,Verzinken (=Produzent 2),ELEO Lager,Annahme Lagerbestand Status Quo,producer
0,9101ub,unbeschichtet,0,0,2.0,0,3
1,9101fz,feuerverzinkt,0,2,3.0,0,3
2,9101dx,DB 703,2,0,3.0,0,3
3,9101sf,Sonderfarben,3,0,2.4,0,3
4,9102ub,unbeschichtet,0,0,2.0,0,2
...,...,...,...,...,...,...,...
87,9318ub,unbeschichtet,0,0,2.0,0,1
88,9316sf,Sonderfarben,3,0,2.4,0,1
89,9316dx,pulverbeschichtet DB703,2,0,3.0,0,1
90,9316fz,feuerverzinkt,0,1,2.0,0,1


In [138]:
def get_cleaned_production_time(df):

    ## Handle lists in SKU col
    df["SKU"] = df["SKU"].apply(str).str.replace(" ", "")

    # 2. In Liste umwandeln
    df["SKU"] = df["SKU"].str.split(",")

    # 3. Exploden → jede SKU bekommt ihre eigene Zeile
    df = df.explode("SKU")

    ## Extract Producer
    prod_cols = [c for c in df.columns if c.startswith("Produzent")]

    # Index der Spalte finden, die NICHT 0 ist
    # → liefert Namen wie "Produzent 3"
    df["producer_col"] = df[prod_cols].apply(lambda row: row[row != 0].index[0] if (row != 0).any() else None, axis=1)

    # Producer-Nummer extrahieren
    df[PRODUCER_COL_NAME] = df["producer_col"].str.extract(r"(\d+)").fillna(0).astype(int)

    ## Extract Producer Time
    # Duration extrahieren (Wert in der jeweiligen Produzent-Spalte)
    df[DURATION_PRODUCER_COL_NAME] = df.apply(
        lambda row: row[row["producer_col"]] if row[PRODUCER_COL_NAME] != 0 else 0,
        axis=1
    )

    ## Add duration of coating
    df[DURATION_COATING_COL_NAME] = (
            df["Duplexbeschichter"].fillna(0) +
            df["Verzinken (=Produzent 2)"].fillna(0)
    )

    # Delete unecassary cols
    cols_to_delete = prod_cols
    cols_to_delete.extend([
        "producer_col",
        "Annahme Lagerbestand Status Quo",
        "ELEO Lager",
        "Duplexbeschichter",
        "Verzinken (=Produzent 2)"
    ])
    df = df.drop(columns=cols_to_delete)

    df["SKU"] = df["SKU"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

    return df

In [145]:
all_articles = pd.DataFrame()
for sheet in [0, 2, 4]:
    df = pd.read_excel("/Users/ap8c3/IdeaProjects/Hackathon/eleo-mind/packages/backend/files/Produktionszeit_Warenfluss_Hackathon_Teilnehmer.xlsx", sheet_name=sheet)

    cleaned_df = get_cleaned_production_time(df)

    prod_line = 0
    if sheet == 0:
        prod_line = 1
    elif sheet == 2:
        prod_line = 2
    elif sheet == 4:
        prod_line = 3

    cleaned_df[PRODUCTION_LINE_COL_NAME] = prod_line

    all_articles = pd.concat([all_articles, cleaned_df])

all_articles

,SKU,Beschichtung,producer,duration_producer,duration_coating,production_line
0,9101UB,unbeschichtet,3,10,0,1
1,9101FZ,feuerverzinkt,3,10,15,1
2,9101DX,DB 703,3,10,15,1
3,9101SF,Sonderfarben,3,10,15,1
4,9102UB,unbeschichtet,2,20,0,1
...,...,...,...,...,...,...
69,SZ21022,unbeschichtet,1,15,0,3
70,SZ2102SF,Sonderfarbe,1,15,15,3
71,SZ21031,pulverbeschichtet DB703,1,15,15,3
72,SZ21032,unbeschichtet,1,15,0,3
